# 269. Alien Dictionary
**Difficulty:** 🔴 Hard (Premium) · **Topic:** Graph · **LeetCode:** https://leetcode.com/problems/alien-dictionary/

## 💡 Concepts

**Core concept(s):** Turn letter-ordering clues into a graph, then **topological sort** it.

**Why it applies here:** Words sorted in an unknown alphabet reveal order clues: the first differing letter between two adjacent words tells you "this letter comes before that one". Those clues are directed edges; a topological order of the letters is a valid alphabet. A cycle (contradiction) means no valid order.

**Key intuition:** Each adjacent pair of words gives one 'letter A before letter B' rule; order all letters consistent with the rules.

---

### 📚 What is a Graph?
A **graph** is dots (**nodes/vertices**) joined by lines (**edges**). Edges can be **directed** (one-way, like prerequisites) or **undirected** (two-way, like friendships). A **grid** is just a graph where each cell links to its neighbors.
- **In Python:** usually an **adjacency list** — a `dict` mapping each node to the list of nodes it connects to.

### 📚 What is Topological Sort (Kahn's method)?
For a directed graph with no cycles, a **topological order** lists nodes so every arrow points forward (do prerequisites first). **Kahn's method:** repeatedly take a node with no remaining incoming arrows, output it, and remove its outgoing arrows.
- **Complexity:** **O(V + E)**. If you can't output every node, there's a **cycle**.
- **In Python:** an in-degree count per node + a queue of zero-in-degree nodes.

---

**Prerequisite knowledge:**
- Directed graph from ordering clues.
- Topological sort (Kahn's) and cycle detection.

## 📝 Problem

Given words sorted by an unknown alphabet, return a possible letter order (or `""` if the input is contradictory).

**Example**
```
["wrt","wrf","er","ett","rftt"] -> "wertf"
```

### Approach — Build Graph + Kahn's Topological Sort

**Idea:** For each adjacent word pair, find the first differing letter → an edge (earlier → later). Then repeatedly output letters with no remaining "must come before" constraints. Special case: if a longer word is a prefix that comes *before* its own prefix, that's invalid.

**Time:** `O(total characters)`. **Space:** `O(unique letters + edges)`.

In [ ]:
from collections import defaultdict, deque

def alien_order(words):
    graph = defaultdict(set)               # letter -> set of letters that must come AFTER it
    indeg = {c: 0 for w in words for c in w}   # every letter starts with 0 "must come before" rules
    for i in range(len(words) - 1):        # compare each adjacent pair of words
        a, b = words[i], words[i + 1]
        minlen = min(len(a), len(b))
        if len(a) > len(b) and a[:minlen] == b[:minlen]:
            return ""                      # e.g. "abc" before "ab" is impossible -> invalid
        for j in range(minlen):
            if a[j] != b[j]:               # first difference reveals the order clue
                if b[j] not in graph[a[j]]:
                    graph[a[j]].add(b[j]); indeg[b[j]] += 1   # a[j] comes before b[j]
                break                      # only the FIRST difference is a real clue
    q = deque([c for c in indeg if indeg[c] == 0])   # letters with no constraints
    order = []
    while q:
        c = q.popleft(); order.append(c)   # output a letter with nothing before it
        for nxt in graph[c]:
            indeg[nxt] -= 1
            if indeg[nxt] == 0:
                q.append(nxt)
    return "".join(order) if len(order) == len(indeg) else ""   # leftover letters -> a cycle

In [ ]:
# Correctness check (verify the returned order respects every adjacent-word clue)
def respects(words, order):
    if order == "":
        return None
    pos = {c: i for i, c in enumerate(order)}
    for a, b in zip(words, words[1:]):
        for x, y in zip(a, b):
            if x != y:
                if pos[x] > pos[y]:
                    return False
                break
        else:
            if len(a) > len(b):
                return False
    return True

tests = [
    (["wrt","wrf","er","ett","rftt"], True),
    (["z","x"], True),
    (["z","x","z"], "invalid"),                # contradiction -> ""
    (["abc","ab"], "invalid"),
]
for words, exp in tests:
    order = alien_order(words)
    ok = respects(words, order)
    print(f"{words} -> {order!r}")
    if exp == "invalid":
        assert order == "", "should be invalid"
    else:
        assert ok is True, "order violates a clue"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)` / `O(V+E)` | ≈ **2×** |
| `O(n log n)`      | ≈ **2×** (slightly more) |
| `O(n²)`           | ≈ **4×** |

Inputs are shaped to force the worst case while keeping recursion shallow (stars / checkerboards) so nothing overflows the stack.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    # n already-sorted distinct words over a small alphabet -> O(total chars) work
    words = []
    for i in range(n):
        s = ''; x = i
        for _ in range(5):
            s += chr(ord('a') + x % 5); x //= 5
        words.append(s[::-1])
    words.sort()
    return (words,)
solutions = {
    "topo sort O(total chars)": alien_order,
}
sizes = [2000, 4000, 8000, 16000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Turn ordering clues into a graph:** "A before B" facts become directed edges; a valid order is a topological sort.
- **First difference wins:** only the first differing letter of two adjacent words is a real clue.
- **Signal:** "derive an order from comparisons / rankings", "unknown alphabet".
- **Related problems:** Course Schedule II, Sequence Reconstruction, Build order.
- **Common pitfalls:** (1) missing the prefix contradiction ("abc" before "ab"); (2) adding edges past the first difference; (3) forgetting letters that have no constraints.